In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import os

DATA_DIR = r"C:\Users\arnaw\OneDrive\Desktop\transfer_recomender\data"

fw = pd.read_csv(os.path.join(DATA_DIR, "fw_clustered.csv"))
mf = pd.read_csv(os.path.join(DATA_DIR, "mf_clustered.csv"))
df = pd.read_csv(os.path.join(DATA_DIR, "df_clustered.csv"))

print(f"FW: {fw.shape}")
print(f"MF: {mf.shape}")
print(f"DF: {df.shape}")

print(f"\nCluster column in FW: {'cluster' in fw.columns}")
print(f"Cluster column in MF: {'cluster' in mf.columns}")
print(f"Cluster column in DF: {'cluster' in df.columns}")

FW: (720, 223)
MF: (900, 223)
DF: (1022, 223)

Cluster column in FW: True
Cluster column in MF: True
Cluster column in DF: True


In [2]:
# Cell 2: Find Similar Players Function
def find_similar_players(player_name, position, top_n=5):
    
    # select correct dataframe
    if position == 'FW':
        data = fw
    elif position == 'MF':
        data = mf
    else:
        data = df
    
    # get numeric features only
    numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()
    numeric_cols = [c for c in numeric_cols if c != 'cluster']
    
    # find the player
    player_row = data[data['Player'] == player_name]
    
    if player_row.empty:
        print(f"Player '{player_name}' not found!")
        return
    
    # get player's cluster
    player_cluster = player_row['cluster'].values[0]
    print(f"{player_name} → Cluster {player_cluster}")
    
    # filter same cluster
    same_cluster = data[data['cluster'] == player_cluster].copy()
    
    # calculate cosine similarity
    player_stats = player_row[numeric_cols].fillna(0).values
    cluster_stats = same_cluster[numeric_cols].fillna(0).values
    
    scores = cosine_similarity(player_stats, cluster_stats)[0]
    
    # add scores to dataframe
    same_cluster['similarity'] = scores
    
    # sort and remove the player himself
    results = same_cluster[same_cluster['Player'] != player_name]
    results = results.sort_values('similarity', ascending=False).head(top_n)
    
    # show results
    print(f"\nTop {top_n} similar players:\n")
    print(results[['Player', 'Squad', 'similarity']].to_string(index=False))

# Test it!
find_similar_players('Erling Haaland', 'FW')

Erling Haaland → Cluster 4

Top 5 similar players:

         Player       Squad  similarity
      Evanilson Bournemouth    0.993511
   Artem Dovbyk        Roma    0.990933
 Emanuel Emegha  Strasbourg    0.990842
Nicolas Jackson     Chelsea    0.981731
      Hugo Duro    Valencia    0.981099


In [8]:
# Cell 3: Test with multiple players
test_cases = [
    ('Harry Kane', 'FW'),
    ('Kevin De Bruyne', 'MF'),
    ('Virgil van Dijk', 'DF')
]

for player, pos in test_cases:
    print(f"\n{'='*50}")
    find_similar_players(player, pos)
    print()


Harry Kane → Cluster 4

Top 5 similar players:

          Player          Squad  similarity
   Shuto Machino  Holstein Kiel    0.992724
Nicolás González       Juventus    0.992041
   Álvaro García Rayo Vallecano    0.990641
    Raúl Jiménez         Fulham    0.988550
 Philipp Hofmann         Bochum    0.986964


Kevin De Bruyne → Cluster 5

Top 5 similar players:

        Player       Squad  similarity
 Ryan Christie Bournemouth    0.998142
 Julian Brandt    Dortmund    0.996518
    Pape Gueye  Villarreal    0.996249
 Pablo Fornals       Betis    0.996159
Danilo Cataldi  Fiorentina    0.996012


Virgil van Dijk → Cluster 2

Top 5 similar players:

            Player         Squad  similarity
     Amir Rrahmani        Napoli    0.998990
       Kim Min-jae Bayern Munich    0.998963
      Jonathan Tah    Leverkusen    0.998603
Nico Schlotterbeck      Dortmund    0.998266
Alexsandro Ribeiro         Lille    0.998233



In [9]:
# Cell 4: Save similarity results to CSV
def save_similarity_results(player_name, position, top_n=10):
    
    if position == 'FW':
        data = fw
    elif position == 'MF':
        data = mf
    else:
        data = df
    
    numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()
    numeric_cols = [c for c in numeric_cols if c != 'cluster']
    
    player_row = data[data['Player'] == player_name]
    
    if player_row.empty:
        print(f"Player '{player_name}' not found!")
        return None
    
    player_cluster = player_row['cluster'].values[0]
    same_cluster = data[data['cluster'] == player_cluster].copy()
    
    player_stats = player_row[numeric_cols].fillna(0).values
    cluster_stats = same_cluster[numeric_cols].fillna(0).values
    
    scores = cosine_similarity(player_stats, cluster_stats)[0]
    same_cluster['similarity'] = scores
    
    results = same_cluster[same_cluster['Player'] != player_name]
    results = results.sort_values('similarity', ascending=False).head(top_n)
    
    return results[['Player', 'Squad', 'Comp', 'cluster', 'similarity']]

# Test and save
print("Testing save function...")
test = save_similarity_results('Erling Haaland', 'FW')
print(test)
print("\n✅ Similarity function ready for recommender!")

Testing save function...
              Player          Squad                Comp  cluster  similarity
221        Evanilson    Bournemouth  eng Premier League        4    0.993511
195     Artem Dovbyk           Roma          it Serie A        4    0.990933
214   Emanuel Emegha     Strasbourg          fr Ligue 1        4    0.990842
318  Nicolas Jackson        Chelsea  eng Premier League        4    0.981731
200        Hugo Duro       Valencia          es La Liga        4    0.981099
278   Gorka Guruzeta  Athletic Club          es La Liga        4    0.976628
342             Kiké         Alavés          es La Liga        4    0.975062
213     Breel Embolo         Monaco          fr Ligue 1        4    0.971936
305   Borja Iglesias     Celta Vigo          es La Liga        4    0.971307
293      Lucas Höler       Freiburg       de Bundesliga        4    0.970767

✅ Similarity function ready for recommender!
